# Paper B batch 3 — density causal test (6.20): pad WikiANN until it is as sparse as ABSA
Same WikiANN-EN rows, spans and test examples; neutral distractor text is added around the span to reach span-word density 0.30 and 0.10 (unpadded ≈ 0.60). Unpadded controls already exist in `IdiomatorRigor_NER_en_bbpe` (same encoders, seeds, epochs, LR). Question: does the byte-BPE QA−BIO strip gap open as density falls? If yes, sparsity (BIO's O-dominated loss) causes the residual; if it stays ≈0, the ABSA residual is something else.
Runtime → **A100** (T4 works, slower). Cell 4 is a 1-epoch timing dry-run: read its wall time before launching cells 5–7. Every launch skips runs whose `metrics.json` already exists on Drive, so a disconnect is safe — just rerun. **After any code change set `FORCE=1` for the affected block.** Last cell releases the runtime.

In [ ]:
# 1. Setup: Drive + repo + deps + GPU gate
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/repo && git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt datasets
import torch
assert torch.cuda.is_available(), 'CPU-only runtime (happens after restarts) — Runtime > Change runtime type > GPU'
print('Device: cuda —', torch.cuda.get_device_name(0))

In [ ]:
# 2. Preflight: patch pushed, Drive roots durable + writable, unpadded controls present. T0 proves runs are fresh.
import os, json, glob, time, shutil, subprocess
T0 = time.time()
assert os.path.exists('experiments/rigor/run_37_pad_density.py'), 'old commit — push run_37 first'
M = '/content/drive/MyDrive'
ROOT_B = f'{M}/IdiomatorRigor_NER_en_bbpe'                  # unpadded controls (density ~0.60)
LEVELS = {'030': 0.30, '010': 0.10}
PAD = {k: f'{M}/IdiomatorRigor_NER_pad_d{k}' for k in LEVELS}
PAD_OW1 = f'{M}/IdiomatorRigor_NER_pad_d010_ow1'
ENCS = [('roberta-base','roberta',0), ('answerdotai/ModernBERT-base','modernbert',1)]
for r in [*PAD.values(), PAD_OW1]:
    os.makedirs(r, exist_ok=True)
    assert os.path.realpath(r).startswith('/content/drive'), f'{r} is not on Drive'
    open(f'{r}/.probe','w').write('ok'); assert open(f'{r}/.probe').read()=='ok'; os.remove(f'{r}/.probe')
for enc in ('roberta','modernbert'):
    for sysn in ('joint','bio'):
        for s in (42,123,7): assert os.path.exists(f'{ROOT_B}/bbpe/{enc}_{sysn}_s{s}/metrics.json'), f'missing unpadded control {enc}_{sysn}_s{s}'
def readback(root, pat='bbpe/*_s*/test_predictions.jsonl', fresh_only=False):
    for f in sorted(glob.glob(f'{root}/{pat}')):
        mt = os.path.getmtime(f)
        if fresh_only and mt < T0: continue
        print(f'{sum(1 for _ in open(f)):5d} rows  {time.ctime(mt)}  {f[len(root)+1:]}')
print('preflight ok')

In [ ]:
# 3. Build ABSA distractor pool (run_33), then the padded WikiANN splits (run_37). Prints realised density + O_WEIGHT.
for dom in ('restaurants','laptops'):
    r = subprocess.run(['python','experiments/rigor/run_33_absa_data_prep.py','--config',dom,'--out-dir',f'data/absa14_{dom}/Splits','--force'],capture_output=True,text=True)
    print(dom, r.stdout[-200:], r.stderr[-300:])
OW = {}
for k, d in LEVELS.items():
    r = subprocess.run(['python','experiments/rigor/run_37_pad_density.py','--density',str(d),'--out-dir',f'data/wikiann_pad_d{k}/Splits','--force'],capture_output=True,text=True)
    print(r.stdout[-900:], r.stderr[-500:])
    m = json.load(open(f'data/wikiann_pad_d{k}/Splits/meta.json')); OW[k] = m['o_weight']
    for s in ('train','dev','test'): assert abs(m['splits'][s]['after']['span_word_frac'] - d) < 0.04, f'{s} density off target'
    assert [m['splits'][s]['after']['n'] for s in ('train','dev','test')] == [m['splits'][s]['before']['n'] for s in ('train','dev','test')]
print('O_WEIGHT by level:', OW, '(unpadded controls ran at 0.615)')

In [ ]:
# 4. Timing dry-run (1 epoch, both trainers) on the sparsest level. NOTE the wall time per encoder to size cells 5-7.
k = '010'
for model, short, trc in ENCS:
    t = time.time()
    !DRIVE_OUT={PAD[k]} DATA_DIR=data/wikiann_pad_d{k}/Splits LANGS=English TEST_LANGS=English O_WEIGHT={OW[k]} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh --dry-run
    print(f'### {short} dry-run (1 epoch joint + 1 epoch bio): {time.time()-t:.0f}s  — a full run is ~7x/6x that per system')
!rm -rf {PAD[k]}/bbpe/_dryrun_joint {PAD[k]}/bbpe/_dryrun_bio

## Density 0.30 and 0.10 — original BIO O-weight (per-level heuristic), RoBERTa + ModernBERT × seeds 42 123 7 × {joint, bio}
Defaults (joint 7 epochs, BIO 6) and LR match the unpadded controls.

In [ ]:
# 5. Launch density 0.30 (12 runs). Resumable.
FORCE = 0
k = '030'
for model, short, trc in ENCS:
    !DRIVE_OUT={PAD[k]} DATA_DIR=data/wikiann_pad_d{k}/Splits LANGS=English TEST_LANGS=English O_WEIGHT={OW[k]} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 6. Launch density 0.10 (12 runs). Resumable.
FORCE = 0
k = '010'
for model, short, trc in ENCS:
    !DRIVE_OUT={PAD[k]} DATA_DIR=data/wikiann_pad_d{k}/Splits LANGS=English TEST_LANGS=English O_WEIGHT={OW[k]} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 7. Density 0.10 at O_WEIGHT=1.0: copy the joint runs (weight-independent), train BIO only (6 runs). Resumable. Separates weight from density.
FORCE = 0
k = '010'
for model, short, trc in ENCS:
    for s in (42,123,7):
        src, dst = f'{PAD[k]}/bbpe/{short}_joint_s{s}', f'{PAD_OW1}/bbpe/{short}_joint_s{s}'
        assert os.path.exists(f'{src}/metrics.json'), f'missing joint source {src} — cell 6 incomplete'
        if not os.path.exists(f'{dst}/metrics.json'): shutil.copytree(src, dst, dirs_exist_ok=True)
    !DRIVE_OUT={PAD_OW1} DATA_DIR=data/wikiann_pad_d{k}/Splits LANGS=English TEST_LANGS=English O_WEIGHT=1.0 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 8. Read back (rows must match joint vs bio) then score each root with NEW output names; print gap-vs-density table incl. unpadded controls.
roots = {'d0.30': PAD['030'], 'd0.10': PAD['010'], 'd0.10 ow1': PAD_OW1}
for tag, root in roots.items():
    print(f'\n##### {tag}'); readback(root)
    for _, short, _ in ENCS:
        for s in (42,123,7):
            nj = sum(1 for _ in open(f'{root}/bbpe/{short}_joint_s{s}/test_predictions.jsonl')); nb = sum(1 for _ in open(f'{root}/bbpe/{short}_bio_s{s}/test_predictions.jsonl'))
            assert nj == nb == 500, f'{tag} {short} s{s}: joint {nj} vs bio {nb} rows (expect 500)'
    out = f"{root}/run_08_pad_{tag.replace(' ','_')}_3seed.json"
    r = subprocess.run(['python','experiments/rigor/run_08_extended_gold.py','--preds-root',root,'--seeds','42','123','7','--out',out,'--force'],capture_output=True,text=True,env={**os.environ,'BBPE':'1'})
    print(r.stdout[-400:], r.stderr[-400:])
print('\n=== strip-mode QA−BIO gap [95% CI] by density (byte-BPE, English, 3 seeds) ===')
def row(path, enc):
    x = json.load(open(path))['encoders'][enc]['langs']['English']['strip']; return f"{x['gap']:+.3f} [{x['gap_ci'][0]:+.3f},{x['gap_ci'][1]:+.3f}]"
ctrl = {'roberta': f'{ROOT_B}/run_08_ner_en_bbpe_3seed.json', 'modernbert': f'{ROOT_B}/run_08_ner_en_bbpe_modernbert_3seed.json'}
for _, short, _ in ENCS:
    print(f'{short:11} unpadded (~0.60): {row(ctrl[short], short)}')
    for tag, root in roots.items(): print(f'{short:11} {tag:10}      : {row(f"{root}/run_08_pad_{tag.replace(chr(32),chr(95))}_3seed.json", short)}')

In [ ]:
# FINAL: prove everything is on Drive, flush, then release the runtime.
# "Run all" stops at the first error, so this only runs if every cell above succeeded.
missing = []
for root in [*PAD.values(), PAD_OW1]:
    for _, short, _ in ENCS:
        for sysn in ('joint','bio'):
            for s in (42,123,7): missing.append(f'{root}/bbpe/{short}_{sysn}_s{s}')
missing = [d for d in missing if not (os.path.exists(f'{d}/metrics.json') and os.path.getsize(f'{d}/test_predictions.jsonl') > 0)]
assert not missing, f'{len(missing)} runs not persisted to Drive, NOT disconnecting: {missing[:5]}'
print('✓ all expected runs persisted on Drive')
drive.flush_and_unmount()   # sync pending Drive writes before the VM disappears
print('Drive flushed — releasing runtime')
from google.colab import runtime
runtime.unassign()